# Tugas Mandiri Pertemuan 12: Prediksi Ulasan Positif Pelanggan Kopi Nusantara

**Nama:** Hilmi Mufid | **NPM:** 2505060046 | **Program Studi:** S1 Teknologi Informasi

**Skenario:** Kopi Nusantara ingin memprediksi apakah seorang pelanggan akan memberikan **ulasan positif (rating 4 ke atas)** berdasarkan atribut pesanannya, agar kombinasi menu atau cabang yang berisiko mengecewakan pelanggan dapat dikenali lebih awal. Notebook ini membangun ML Pipeline klasifikasi dengan Spark MLlib.

### Menyiapkan Dataset

Dataset dibuat dengan cell dari modul (3000 baris, `seed=212`) dan disimpan sebagai `tugas12_ulasan.csv`.

In [2]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


Membuat `SparkSession`, lalu membaca `tugas12_ulasan.csv`:

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, min as spark_min, max as spark_max
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import os

spark = SparkSession.builder \
    .appName("Tugas12-KlasifikasiKopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

# Dataset dibaca dari disk lokal (awalan file:// mencegah Spark mencarinya di HDFS)
path_csv = "file://" + os.path.abspath("tugas12_ulasan.csv")
df_tugas = spark.read.csv(path_csv, header=True, inferSchema=True)

print("Jumlah baris:", df_tugas.count())
df_tugas.printSchema()
print("Distribusi label:")
df_tugas.groupBy("ulasan_positif").count().show()

SparkSession siap. Versi Spark: 3.5.9
Jumlah baris: 3000
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

Distribusi label:
+--------------+-----+
|ulasan_positif|count|
+--------------+-----+
|             1| 1390|
|             0| 1610|
+--------------+-----+



## A. Membangun ML Pipeline

Pipeline terdiri dari enam tahap: tiga `StringIndexer`, satu `OneHotEncoder`, satu `VectorAssembler` (menggabungkan hasil one hot encoding dengan `qty` dan `waktu_tunggu_menit`), dan `LogisticRegression` dengan `labelCol="ulasan_positif"`.

In [5]:
# Tahap 1 sampai 3: StringIndexer untuk setiap kolom kategorikal
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# Tahap 4: OneHotEncoder untuk ketiga kolom indeks sekaligus
ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

# Tahap 5: VectorAssembler menggabungkan hasil one hot encoding dengan qty dan waktu_tunggu_menit
assembler = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

# Tahap 6: Logistic Regression dengan target ulasan_positif
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])
print("Pipeline Logistic Regression terdefinisi dengan", len(pipeline_lr.getStages()), "tahap (belum dijalankan).")

Pipeline Logistic Regression terdefinisi dengan 6 tahap (belum dijalankan).


## B. Pelatihan dan Evaluasi

Data dibagi 80:20 (`seed=77`), pipeline dilatih, lalu dievaluasi dengan accuracy, F1 score, dan AUC ROC, ditambah confusion matrix sederhana dengan `groupBy`.

In [7]:
train, test = df_tugas.randomSplit([0.8, 0.2], seed=77)
print(f"Data training: {train.count()} | Data testing: {test.count()}")

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(pred_lr)
f1_lr = ev_f1.evaluate(pred_lr)
auc_lr = ev_auc.evaluate(pred_lr)

print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("Confusion matrix Logistic Regression:")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Data training: 2411 | Data testing: 589
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650
Confusion matrix Logistic Regression:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



## C. Model Pembanding: Random Forest

`RandomForestClassifier` (`numTrees=30`) memakai lima tahap feature engineering yang identik dengan bagian A, hanya tahap terakhir yang diganti.

In [8]:
# Hanya tahap terakhir yang diganti, lima tahap awal identik dengan bagian A
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(pred_rf)
f1_rf = ev_f1.evaluate(pred_rf)
auc_rf = ev_auc.evaluate(pred_rf)

print("=== Perbandingan Hasil ===")
print(f"Logistic Regression : Accuracy {acc_lr:.4f} | F1 {f1_lr:.4f} | AUC {auc_lr:.4f}")
print(f"Random Forest       : Accuracy {acc_rf:.4f} | F1 {f1_rf:.4f} | AUC {auc_rf:.4f}")

=== Perbandingan Hasil ===
Logistic Regression : Accuracy 0.7929 | F1 0.7926 | AUC 0.8650
Random Forest       : Accuracy 0.7980 | F1 0.7975 | AUC 0.8565


**Catatan perbandingan:** kedua model berimbang pada accuracy dan F1, tetapi Logistic Regression umumnya unggul tipis pada AUC. Hal ini masuk akal karena label pada dataset ini dibentuk dari hubungan yang sederhana dan hampir lurus dengan `waktu_tunggu_menit`, sehingga model sederhana sudah cukup, sementara Random Forest lebih mudah ikut mempelajari variasi acak pada data latih.

## D. Analisis Kesalahan Model

Model terbaik dipilih otomatis berdasarkan AUC tertinggi, lalu baris yang salah diprediksi (`ulasan_positif != prediction`) diperiksa untuk mencari pola.

In [9]:
# Memilih model terbaik berdasarkan AUC tertinggi
if auc_lr >= auc_rf:
    nama_terbaik, pred_terbaik = "Logistic Regression", pred_lr
else:
    nama_terbaik, pred_terbaik = "Random Forest", pred_rf
print("Model terbaik berdasarkan AUC:", nama_terbaik)

# Baris yang salah diprediksi (label berbeda dengan prediction)
salah = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))
print("Jumlah prediksi salah:", salah.count(), "dari", pred_terbaik.count(), "data testing")

salah.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction").show(10)

# Ringkasan pola: rata rata, minimum, dan maksimum waktu tunggu pada baris yang salah
salah.agg(
    avg("waktu_tunggu_menit").alias("rata_rata_tunggu"),
    spark_min("waktu_tunggu_menit").alias("min_tunggu"),
    spark_max("waktu_tunggu_menit").alias("max_tunggu")
).show()

# Jumlah kesalahan untuk setiap nilai waktu tunggu
salah.groupBy("waktu_tunggu_menit").count().orderBy("waktu_tunggu_menit").show(25)

Model terbaik berdasarkan AUC: Logistic Regression
Jumlah prediksi salah: 122 dari 589 data testing
+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|  cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|      Malam|  1|                15|             1|       0.0|
|Americano|Magelang|      Siang|  2|                22|             1|       0.0|
|Americano|Magelang|       Sore|  1|                 7|             0|       1.0|
|Americano|Magelang|       Sore|  2|                11|             0|       1.0|
|Americano|Magelang|       Sore|  3|                 5|             0|       1.0|
|Americano|Magelang|       Sore|  3|                13|             1|       0.0|
|Americano|Magelang|       Sore|  5|                 6|             0|       1.0|
|Americano|Magelang|       Sore|  5|                15|             1|       0.0

**Pola yang terlihat:** kesalahan tidak tersebar merata, melainkan paling banyak terjadi pada waktu tunggu sekitar 9 sampai 15 menit, yaitu di sekitar ambang 12 menit. Di rentang ini peluang ulasan positif dan negatif hampir seimbang (mendekati 50:50), sehingga tebakan paling sulit. Pada waktu tunggu yang sangat singkat atau sangat lama, model hampir selalu benar.

Untuk mendukung kesimpulan bisnis, koefisien model Logistic Regression untuk fitur numerik ditampilkan:

In [10]:
# Koefisien Logistic Regression untuk fitur numerik. Urutan vektor fitur: 10 kolom one hot encoding
# (5 menu, 2 cabang, 3 waktu pesan), lalu qty, lalu waktu_tunggu_menit
model_akhir = model_lr.stages[-1]
koef = model_akhir.coefficients
print("Koefisien one hot (menu, cabang, waktu pesan):", [round(float(x), 3) for x in koef[:10]])
print("Koefisien qty                :", round(koef[10], 4))
print("Koefisien waktu_tunggu_menit :", round(koef[11], 4))
print("Intercept                    :", round(model_akhir.intercept, 4))

Koefisien one hot (menu, cabang, waktu pesan): [0.117, 0.019, -0.086, -0.144, -0.019, 0.065, -0.114, -0.065, 0.253, 0.186]
Koefisien qty                : -0.0108
Koefisien waktu_tunggu_menit : -0.222
Intercept                    : 2.6615


## E. Kesimpulan Bisnis

Model Logistic Regression yang dibangun menunjukkan bahwa **waktu tunggu** adalah faktor yang paling jelas memengaruhi ulasan positif. Koefisien `waktu_tunggu_menit` bernilai negatif (sekitar minus 0,23 per menit), sedangkan koefisien `qty` mendekati nol, dan koefisien menu, cabang, serta waktu pesan kecil (umumnya di bawah 0,3) tanpa pola yang konsisten. Dari kurva yang dipelajari model, peluang ulasan positif sekitar 85% pada waktu tunggu 5 menit, turun menjadi sekitar 50% pada 12 menit, dan tinggal sekitar 15% pada 20 menit. Rekomendasi pertama, manajemen sebaiknya menetapkan target waktu tunggu maksimal sekitar 10 menit dan memantaunya di setiap cabang, karena mempercepat layanan lebih berdampak daripada mengubah menu atau jadwal. Kedua, hasil analisis kesalahan menunjukkan prediksi paling sering meleset pada waktu tunggu sekitar 9 sampai 15 menit, yaitu daerah abu abu di mana pelanggan bisa puas atau kecewa. Pada rentang inilah tindakan proaktif seperti permintaan maaf, kompensasi kecil, atau pembaruan status pesanan paling bermanfaat. Ketiga, model dapat dipakai sebagai sistem peringatan dini untuk pesanan yang waktu tunggunya mulai melewati 12 menit.

Menutup `SparkSession`:

In [11]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
